# 04 - XGBoost on TF-IDF + rule features (Person 2)

Training was run with `python -m src.models.xgboost_model --train ... --val ... --test ...`, which saves the model and
`results/`. This notebook **loads those saved results**, shows them, and does the error analysis. It does not retrain.

In [ ]:
# --- change these paths to match where your files are ---
REPO = "/content/p2"
TEST = "/content/test.csv"
RESULTS = REPO + "/results"
MODEL_DIR = REPO + "/models/xgboost"

import sys, json
sys.path.insert(0, REPO)

## Results

In [ ]:
import numpy as np, pandas as pd
from IPython.display import display, Image

M = json.load(open(f"{RESULTS}/xgboost_metrics.json"))
cols = ["n", "n_fraud", "precision", "recall", "f1", "pr_auc", "roc_auc", "tp", "fp", "fn", "tn"]
table = pd.DataFrame({"validation": M["val"], "test (all rows)": M["test"],
                      "test (unseen text)": M["test_unseen_text"],
                      "test @ threshold 0.5": M["test_at_0.5"]}).T[cols]
display(table)
print("tuned params:", M["best_params"], "| trees:", M["best_iteration"], "| features:", M["n_features"])

### Ablation: which features carry the model? (test set)

In [ ]:
ab = pd.DataFrame(M["ablation_test"]).T[["precision", "recall", "f1", "pr_auc"]]
ab.loc["both (final model)"] = [M["test"][k] for k in ["precision", "recall", "f1", "pr_auc"]]
display(ab)

In [ ]:
for f in ["xgboost_pr_curve.png", "xgboost_confusion_matrix.png", "feature_importance.png"]:
    display(Image(f"{RESULTS}/{f}"))

## Error analysis
Which fraud postings does the model miss, and which real ones does it flag?

In [ ]:
from src.models.xgboost_model import load_artifacts, build_matrix, predict
from src.features.rule_features import extract_features_df

te = pd.read_csv(TEST)
art = load_artifacts(MODEL_DIR)
Xte = build_matrix(extract_features_df(te), art["tfidf"].transform(te["text"].fillna("")))
te["prob"] = art["model"].predict_proba(Xte)[:, 1]
te["pred"] = (te["prob"] >= art["meta"]["threshold"]).astype(int)

fraud = te[te["fraudulent"] == 1].assign(
    caught=lambda d: d["pred"],
    no_profile=lambda d: d["company_profile"].isna(),
    desc_len=lambda d: d["description"].fillna("").str.len())
display(fraud.groupby("caught").agg(
    n=("prob", "size"), no_profile=("no_profile", "mean"), has_logo=("has_company_logo", "mean"),
    has_salary=("salary_range", lambda s: s.notna().mean()), median_desc_len=("desc_len", "median"),
    mean_prob=("prob", "mean")).round(3))

In [ ]:
print("Missed fraud, lowest scores first:")
display(fraud[fraud["caught"] == 0].sort_values("prob").head(10)[["title", "prob", "has_company_logo"]])

fp = te[(te["fraudulent"] == 0) & (te["pred"] == 1)]
print(f"False alarms: {len(fp)} | no company profile: {fp['company_profile'].isna().mean():.0%} "
      f"| has logo: {fp['has_company_logo'].mean():.0%}")
display(fp.sort_values("prob", ascending=False).head(10)[["title", "prob", "has_company_logo"]])

## The `predict(text)` interface used by the app

In [ ]:
scam = ("URGENT! Work from home, earn up to $500 per day, no experience needed. "
        "Pay a small registration fee and contact us on WhatsApp or hiring.team@gmail.com.")
real = ("Senior Software Engineer: 5 years of Python and SQL, join our payments platform team. "
        "Benefits: health insurance, 25 days leave.")
for name, text in (("scam-style", scam), ("normal", real)):
    print(name, "->", predict(text, model_dir=MODEL_DIR))

## Notes for the report
- Compare the table above with Person 1's baseline using the same columns, including the **unseen text** row.
- The scam-style text may be rated "real": the labelled data contains almost no scam wording, so the model relies mostly on
  missing company profile / logo and on template phrases. The `reasons` list still shows the rule signals that fired.
- Pasted text has no company profile / logo fields, which removes the strongest feature.